# Proyecto Integrador - Template

Usa este notebook como punto de partida para tu proyecto final.

## Información del proyecto
- **Nombre**: [Tu proyecto]
- **Dataset**: [Fuente y descripción]
- **Problema**: [Clasificación binaria / Regresión]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
)
import xgboost as xgb
import joblib

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---
## 1. Carga de datos

In [ ]:
# Cargar tu dataset
# df = pd.read_csv('tu_dataset.csv')

# Ejemplo con datos sintéticos (reemplazar con tu dataset)
df = pd.DataFrame({
    'feature_1': np.random.randn(1000),
    'feature_2': np.random.randn(1000),
    'target': np.random.randint(0, 2, 1000)
})

print(f"Shape: {df.shape}")
df.head()

---
## 2. EDA (Análisis Exploratorio)

In [ ]:
# Información general
print(f"Dimensiones: {df.shape}")
print(f"\nTipos de datos:")
print(df.dtypes)
print(f"\nValores nulos:\n{df.isnull().sum()}")
print(f"\nDistribución del target:\n{df['target'].value_counts(normalize=True)}")

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Visualizaciones
# TODO: Agregar histogramas, boxplots, correlaciones
pass

---
## 3. Preparación de datos

In [ ]:
# Definir features y target
target = 'target'
features = [c for c in df.columns if c != target]

# Split
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

In [ ]:
# Encoding
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))
X_test = dv.transform(df_test[features].to_dict(orient='records'))

print(f"Features codificadas: {X_train.shape[1]}")

---
## 4. Modelado

In [ ]:
# Modelo 1: Logística (baseline)
lr = LogisticRegression(solver='liblinear', max_iter=1000)
lr.fit(X_train, y_train)
auc_lr = roc_auc_score(y_val, lr.predict_proba(X_val)[:, 1])
print(f"Logística AUC: {auc_lr:.4f}")

In [ ]:
# Modelo 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
auc_rf = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
print(f"Random Forest AUC: {auc_rf:.4f}")

In [ ]:
# Modelo 3: XGBoost
feature_names = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 4,
    'learning_rate': 0.1,
    'verbosity': 0,
}

xgb_model = xgb.train(
    params, dtrain,
    num_boost_round=200,
    evals=[(dval, 'val')],
    early_stopping_rounds=20,
    verbose_eval=False
)
auc_xgb = roc_auc_score(y_val, xgb_model.predict(dval))
print(f"XGBoost AUC: {auc_xgb:.4f}")

In [ ]:
# Comparación
print("\n" + "=" * 30)
print("COMPARACIÓN DE MODELOS")
print("=" * 30)
print(f"{'Modelo':<20} {'AUC':>6}")
print("-" * 28)
print(f"{'Logística':<20} {auc_lr:>6.4f}")
print(f"{'Random Forest':<20} {auc_rf:>6.4f}")
print(f"{'XGBoost':<20} {auc_xgb:>6.4f}")

---
## 5. Evaluación del mejor modelo

In [ ]:
# Seleccionar mejor modelo y evaluar en TEST
# TODO: Seleccionar tu mejor modelo y entrenar con train+val

X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

mejor_modelo = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
mejor_modelo.fit(X_full, y_full)

y_test_proba = mejor_modelo.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= 0.5).astype(int)

print("EVALUACIÓN FINAL (TEST)")
print("-" * 30)
print(f"AUC:       {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_test_pred):.4f}")

In [ ]:
# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, ax=ax, cmap='Blues')
plt.title('Matriz de Confusión - Test')
plt.show()

---
## 6. Serializar modelo

In [ ]:
# Guardar para el servicio
joblib.dump(mejor_modelo, '../servicio/modelo.joblib')
joblib.dump(dv, '../servicio/vectorizer.joblib')

print("Modelo y vectorizer guardados en servicio/")
print("\nSiguiente paso: crear main.py con FastAPI y Dockerfile")

---
## 7. Próximos pasos

1. Crear `servicio/main.py` con FastAPI
2. Crear `servicio/Dockerfile`
3. Construir imagen: `docker build -t mi-proyecto .`
4. Ejecutar: `docker run -p 8000:8000 mi-proyecto`
5. Probar: abrir `http://localhost:8000/docs`